# 03 — Hyperparameter Tuning

Runs `RandomizedSearchCV` to find the best hyperparameters for each
stellar parameter (Teff, logg, [Fe/H]).

**Feature selection:**
- Set `USE_FEATURE_SELECTION = False` to use all available features
- Set `USE_FEATURE_SELECTION = True` to load the top features saved by `02_feature_importance.ipynb`

**Output:** one `.joblib` pipeline per parameter in the `pipelines/` folder

## Configuration

In [ ]:
import os
import json
import pandas as pd
from datetime import datetime
from tqdm.notebook import tqdm
import minas as mg

# ── Files ─────────────────────────────────────────────────────────────────────
INPUT_FILE   = 'input_catalog.csv'  # output from 01_data_creation.ipynb
INDEX_COL    = 'ID'                 # unique identifier column (or None)
PIPELINE_DIR = 'pipelines'          # folder where trained pipelines are saved

# ── Survey ────────────────────────────────────────────────────────────────────
SURVEY = 'SPLUS'   # 'SPLUS', 'JPLUS', 'JPAS', etc.

# ── Parameters to tune ───────────────────────────────────────────────────────
PARAM_LIST = ['teff', 'logg', 'feh']

# ── Distance column (set to None if absolute magnitudes are not needed) ───────
DIST_COL = 'Dist'

# ── Model type ────────────────────────────────────────────────────────────────
MODEL_TYPE = 'XGB'   # 'RF' or 'XGB'

# ── Feature selection ─────────────────────────────────────────────────────────
# False → use all features from work_df
# True  → load top features from JSON files saved by 02_feature_importance.ipynb
USE_FEATURE_SELECTION = False
FEATURES_DIR          = 'features'  # folder with <param>_features.json files

# ── Search settings ───────────────────────────────────────────────────────────
N_ITER       = 20    # number of random hyperparameter combinations to try
CV           = 3     # cross-validation folds
TEST_SIZE    = 0.25
N_JOBS       = -1    # -1 = use all CPUs
RANDOM_STATE = 42

os.makedirs(PIPELINE_DIR, exist_ok=True)
print(f'Model          : {MODEL_TYPE}')
print(f'Feature select.: {USE_FEATURE_SELECTION}')
print(f'n_iter / cv    : {N_ITER} / {CV}')

## Hyperparameter grid

Edit the dictionary below to define the search space.
Keys must be prefixed with the pipeline step name.

- **RF** pipeline steps: `imputer__`, `selectkbest__` (if used), `randomforestregressor__`
- **XGB** pipeline steps: `selectkbest__` (if used), `xgbregressor__`

If `USE_FEATURE_SELECTION = False`, remove or ignore `selectkbest__k`.

In [ ]:
# ── XGBoost search space ──────────────────────────────────────────────────────
if MODEL_TYPE == 'XGB':
    param_dist = {
        'xgbregressor__n_estimators'    : [100, 300, 500],
        'xgbregressor__max_depth'       : [3, 6, 10],
        'xgbregressor__learning_rate'   : [0.01, 0.05, 0.1],
        'xgbregressor__subsample'       : [0.7, 0.8, 1.0],
        'xgbregressor__colsample_bytree': [0.7, 0.8, 1.0],
        'xgbregressor__gamma'           : [0, 0.1, 0.5],
    }

# ── Random Forest search space ────────────────────────────────────────────────
elif MODEL_TYPE == 'RF':
    param_dist = {
        'randomforestregressor__n_estimators'    : [100, 300, 500],
        'randomforestregressor__max_features'    : ['sqrt', 'log2'],
        'randomforestregressor__min_samples_leaf': [1, 5, 10],
        'randomforestregressor__bootstrap'       : [True, False],
    }

# ── Add SelectKBest if using feature selection ────────────────────────────────
if USE_FEATURE_SELECTION:
    # k values will be set automatically based on the number of selected features
    param_dist['selectkbest__k'] = [5, 10, 15, 20]

print('Search space:')
for k, v in param_dist.items():
    print(f'  {k}: {v}')

## Step 1 — Load catalog and assemble feature DataFrames

In [ ]:
def get_column(df, aliases):
    """Return the first matching column from a list of aliases."""
    for col in aliases:
        if col in df.columns:
            return col
    raise KeyError(f'No column found for aliases: {aliases}')


df_base = pd.read_csv(INPUT_FILE)
if INDEX_COL and INDEX_COL in df_base.columns:
    df_base = df_base.set_index(INDEX_COL)

filters = mg.FILTERS[SURVEY]
print(f'Catalog loaded: {len(df_base):,} objects')

preprocessed = {}
for param in PARAM_LIST:
    df = df_base.copy()

    # Remove invalid values
    param_col = get_column(df, mg.PARAM_ALIASES[param])
    df = df[df[param_col] != -9999].dropna(subset=[param_col])

    # Compute absolute magnitudes
    if DIST_COL and DIST_COL in df.columns:
        df = mg.preprocess.calculate_abs_mag(df, filters, DIST_COL)

    # Assemble feature DataFrame
    work_df = mg.preprocess.assemble_work_df(
        df=df,
        filters=filters,
        correction_pairs=None,
        add_colors=True,
        verbose=False,
    )

    # Apply feature selection if requested
    if USE_FEATURE_SELECTION:
        feat_path = os.path.join(FEATURES_DIR, f'{param}_features.json')
        if not os.path.exists(feat_path):
            raise FileNotFoundError(
                f"Feature file not found: {feat_path}\n"
                f"Run 02_feature_importance.ipynb first."
            )
        with open(feat_path) as f:
            selected = json.load(f)
        work_df = work_df[selected]
        print(f'  [{param:4s}] {len(df):,} objects  |  {work_df.shape[1]} features (selected)')
    else:
        print(f'  [{param:4s}] {len(df):,} objects  |  {work_df.shape[1]} features (all)')

    preprocessed[param] = {'df': df, 'work_df': work_df, 'param_col': param_col}

## Step 2 — Run hyperparameter search

In [ ]:
datetime_str = datetime.now().strftime('%Y%m%d%H%M%S')
tuning_ids   = []

for param in tqdm(PARAM_LIST, desc='Tuning parameters'):
    work_df   = preprocessed[param]['work_df']
    df        = preprocessed[param]['df']
    param_col = preprocessed[param]['param_col']

    tuning_id = f'{datetime_str}_{SURVEY}_{param}_{MODEL_TYPE}'

    best_pipeline, search = mg.hyperparameter_search(
        X=work_df,
        Y=df[param_col],
        model_type=MODEL_TYPE,
        param_dist=param_dist,
        tuning_id=tuning_id,
        n_iter=N_ITER,
        cv=CV,
        test_size=TEST_SIZE,
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
        save_dir=PIPELINE_DIR,
    )

    tuning_ids.append(tuning_id)
    print(f'  [{param:4s}] best R2 = {search.best_score_:.4f}  |  id: {tuning_id}')
    print(f'         best params: {search.best_params_}')

print(f'\nPipelines saved to: {PIPELINE_DIR}/')
print(f'Tuning IDs:')
for tid in tuning_ids:
    print(f'  {tid}')